# FastAPI — Pydantic Models & Schemas

Interactive notebook companion to `03_pydantic_schemas.md`

In [ ]:
from pydantic import BaseModel, Field, EmailStr, field_validator, model_validator
from typing import Optional
from datetime import datetime
from enum import Enum

In [ ]:
# Basic Pydantic model
class UserCreate(BaseModel):
    username: str = Field(min_length=3, max_length=50)
    email: str
    age: Optional[int] = Field(default=None, ge=0, le=150)

# Valid
user = UserCreate(username='alice', email='alice@example.com', age=30)
print('Valid user:', user.model_dump())

# Invalid — username too short
try:
    UserCreate(username='ab', email='test@test.com')
except Exception as e:
    print('Validation error:', e.errors()[0]['msg'])

In [ ]:
# Custom validators
import re

class PasswordChange(BaseModel):
    current_password: str
    new_password: str
    confirm_password: str

    @field_validator('new_password')
    @classmethod
    def password_strength(cls, v):
        if len(v) < 8:
            raise ValueError('Must be at least 8 characters')
        if not re.search(r'[A-Z]', v):
            raise ValueError('Must contain uppercase letter')
        if not re.search(r'[0-9]', v):
            raise ValueError('Must contain a digit')
        return v

    @model_validator(mode='after')
    def passwords_match(self):
        if self.new_password != self.confirm_password:
            raise ValueError('Passwords do not match')
        return self

# Valid
pc = PasswordChange(current_password='old', new_password='NewPass1', confirm_password='NewPass1')
print('Valid:', pc.model_dump())

# Mismatch
try:
    PasswordChange(current_password='old', new_password='NewPass1', confirm_password='Different1')
except Exception as e:
    print('Error:', e.errors()[0]['msg'])

In [ ]:
# Schema inheritance pattern
class ProductBase(BaseModel):
    name: str
    price: float = Field(gt=0)

class ProductCreate(ProductBase):
    stock: int = 0

class ProductUpdate(BaseModel):
    name: Optional[str] = None
    price: Optional[float] = Field(default=None, gt=0)
    stock: Optional[int] = None

class ProductResponse(ProductBase):
    id: int
    stock: int
    model_config = {'from_attributes': True}

# Demonstrate model_dump with exclude_unset
update = ProductUpdate(price=19.99)  # Only price set
print('All fields:', update.model_dump())
print('Only set:', update.model_dump(exclude_unset=True))  # Only price

In [ ]:
# from_attributes — ORM model → Pydantic
class FakeORMUser:
    '''Simulates a SQLAlchemy ORM object'''
    def __init__(self, id, username, email):
        self.id = id
        self.username = username
        self.email = email
        self.hashed_password = 'secret_hash'  # Should NOT appear in response

class UserResponse(BaseModel):
    id: int
    username: str
    email: str
    # hashed_password NOT included
    model_config = {'from_attributes': True}

orm_user = FakeORMUser(1, 'alice', 'alice@example.com')
response = UserResponse.model_validate(orm_user)
print('Response (no password):', response.model_dump())